In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
import torch

# Set a random seed for reproducibility
set_seed(42)

# Load the gpt2-medium tokenizer and model
tokenizer = AutoTokenizer.from_pretrained('gpt2-medium')
model = AutoModelForCausalLM.from_pretrained('gpt2-medium')

print("Model and tokenizer loaded successfully!")

config.json:   0%|          | 0.00/718 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.52G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2-medium
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model and tokenizer loaded successfully!


In [ ]:
from transformers import pipeline, set_seed
generator = pipeline('text-generation', model='gpt2-large')
set_seed(42)
generator("Hello, I'm a language model,", max_length=30, num_return_sequences=5)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.25G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/436 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2-large
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...35}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'num_return_sequences', 'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=30) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': "Hello, I'm a language model, so I can tell you that some languages are much more expressive than others.\n\nBut even in the most expressive languages, there's still a ton of stuff that's hard to express, and in the end, that's what matters - the expressive power of your language.\n\nSo in this talk, I want to share some of what I think is the most useful pieces of data I've collected, and how they can help you learn about your own language.\n\nSo let's get started.\n\nThe first thing I want to do is introduce you to the language of this talk, Rust.\n\nRust is a modern, statically typed programming language for the Rust programming language.\n\nIt's a fun language, and I think it's the perfect introduction to Rust and Rust programming.\n\nAll right, let's go.\n\nSo what is Rust?\n\nRust is a modern, statically typed programming language, based on the C-family of languages.\n\nIt's an OCaml-like language with a rich type system that allows you to express almost anyth

In [ ]:
def generate_filtered_response(user_question, model, tokenizer, max_length=100):
    """
    Generates a response to a user question, filtering for Python coding questions.

    Args:
        user_question (str): The question asked by the user.
        model: The pre-trained language model (e.g., GPT-2).
        tokenizer: The tokenizer corresponding to the model.
        max_length (int): Maximum length of the generated response.

    Returns:
        str: The generated response or a predefined message if not a coding question.
    """
    if is_python_coding_question(user_question, model, tokenizer):
        # Construct prompt for code generation
        generation_prompt = f"Write Python code to {user_question}"

        # Encode the prompt
        inputs = tokenizer.encode(generation_prompt, return_tensors='pt')

        # Generate response
        output = model.generate(
            inputs,
            max_length=max_length,
            num_return_sequences=1,
            pad_token_id=tokenizer.eos_token_id
        )

        # Decode and return the generated text
        generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
        return generated_text
    else:
        return "I can only answer questions related to Python coding. Please ask a Python coding question."


In [ ]:
def is_python_coding_question(user_question, model, tokenizer):
    """
    Classifies if a user question is a Python coding question using a pre-trained model.

    Args:
        user_question (str): The question asked by the user.
        model: The pre-trained language model (e.g., GPT-2).
        tokenizer: The tokenizer corresponding to the model.

    Returns:
        bool: True if the question is a Python coding question, False otherwise.
    """
    # Create a prompt to guide the model for classification using few-shot examples
    classification_prompt = (
        "Is the following question a Python coding question? Answer 'yes' or 'no'.\n\n"
        "Question: write a function to sort a list\n"
        "Answer: yes\n\n"
        "Question: What is the capital of France?\n"
        "Answer: no\n\n"
        "Question: create a list comprehension for squares\n"
        "Answer: yes\n\n"
        "Question: How tall is the Eiffel Tower?\n"
        "Answer: no\n\n"
        f"Question: {user_question}\n"
        f"Answer:"
    )

    # Encode the prompt
    inputs = tokenizer.encode(classification_prompt, return_tensors='pt')

    # Generate a short response (just 'yes' or 'no')
    output = model.generate(
        inputs,
        max_new_tokens=5, # Limit output length to just get 'yes' or 'no'
        num_return_sequences=1,
        pad_token_id=tokenizer.eos_token_id
    )

    # Decode the response and convert to lowercase for robust checking
    generated_text = tokenizer.decode(output[0][inputs.shape[-1]:], skip_special_tokens=True).strip().lower()
    print(f"DEBUG: Generated text for classification: '{generated_text}'") # Debugging line

    # Check if the generated text contains 'yes'
    return 'yes' in generated_text

In [ ]:
# Test cases
python_question_1 = "write a function to add two numbers"
python_question_2 = "create a list comprehension to get even numbers"
non_coding_question_1 = "What is the capital of France?"
non_coding_question_2 = "Tell me about the history of artificial intelligence."

print("--- Testing Python Coding Questions ---")
response1 = generate_filtered_response(python_question_1, model, tokenizer)
print(f"Question: {python_question_1}\nResponse: {response1}\n")

response2 = generate_filtered_response(python_question_2, model, tokenizer)
print(f"Question: {python_question_2}\nResponse: {response2}\n")

print("--- Testing Non-Coding Questions ---")
response3 = generate_filtered_response(non_coding_question_1, model, tokenizer)
print(f"Question: {non_coding_question_1}\nResponse: {response3}\n")

response4 = generate_filtered_response(non_coding_question_2, model, tokenizer)
print(f"Question: {non_coding_question_2}\nResponse: {response4}\n")

--- Testing Python Coding Questions ---
DEBUG: Generated text for classification: 'yes

question:'
Question: write a function to add two numbers
Response: Write Python code to write a function to add two numbers together.

The function is called add_two_numbers() .

The function takes two arguments:

The number of numbers to add.

The number of arguments to the function.

The function takes the number of arguments and returns the result.

The function takes the number of arguments and returns the result.

The function takes the number of arguments and returns the result.

The function

DEBUG: Generated text for classification: 'yes

question:'
Question: create a list comprehension to get even numbers
Response: Write Python code to create a list comprehension to get even numbers.

import numpy as np import matplotlib.pyplot as plt import matplotlib.pyplot.pyplotlib as plt import matplotlib.pyplot.pyplotlib.plot as plt import matplotlib.pyplot.pyplot.pyplotlib.plot.plot.plot.plot.plot.pl